*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*

> This notebook contains the raw code for Chapter 4: Designing Custom Architectures with torch.nn. To understand how modules, parameters, and buffers are registered and managed in a real model, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

Modules are the building blocks of PyTorch models. They hold state, register parameters and buffers, and describe how inputs flow through a computation graph in a reusable and structured way.

## 1. The `nn.Module` and `nn.Parameter` Lifecycle
### Step 1: Establish the `nn.Module` Contract


In [ ]:
# nn.Module is the base class for all PyTorch layers and models.
# __init__() creates the persistent structure (parameters, buffers, child modules).
# forward() defines the actual computation that executes when you call model(x).
import torch
import torch.nn as nn

class CustomModel(nn.Module):
    # Define a simple feedforward neural network with one hidden layer
    # __init__ method sets up the layers of the model
    def __init__(self, input_size, hidden_size, output_size):
        super(CustomModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    # forward method defines the forward pass of the model
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

x = torch.randn(10, 5)  # Example input
model = CustomModel(input_size=5, hidden_size=10, output_size=2)
out = model(x)
print("Output of the model:", out)

Output of the model: tensor([[ 1.1722e-01, -6.7249e-02],
        [ 2.5275e-01,  9.1169e-02],
        [ 1.5369e-01,  6.2126e-02],
        [ 1.5044e-01,  9.9039e-02],
        [ 2.1412e-01,  6.9577e-02],
        [ 4.3761e-01,  1.0258e-02],
        [ 2.0191e-01,  1.0070e-01],
        [ 2.3294e-01, -3.5961e-04],
        [ 3.2110e-01, -7.2546e-02],
        [ 1.0760e-01,  8.8257e-02]], grad_fn=<AddmmBackward0>)


### Step 2: Register Learnable State with `nn.Parameter`

In [ ]:
import torch
import torch.nn as nn

# nn.Parameter is a Tensor subclass that signals learnable state.
# Only Parameters are exposed to optimizers; plain tensors are not tracked.
# Parameters are automatically found by parameters(), state_dict(), and device transfers.
class CustomOffsetLayer(nn.Module):
    def __init__(self, feature_size):
        super().__init__()  # Initialize internal tracking dictionaries

        # A raw tensor is not registered as a model parameter.
        self.ignored_tensor = torch.randn(feature_size)

        # A Parameter is registered and tracked for optimization.
        self.offset = nn.Parameter(torch.randn(feature_size))

    def forward(self, x):
        return x + self.offset

layer = CustomOffsetLayer(feature_size=128)
sample = torch.randn(4, 128)
output = layer(sample)

parameter_names = dict(layer.named_parameters())
state_keys = layer.state_dict().keys()

print(parameter_names.keys())  # dict_keys(['offset'])
assert output.shape == sample.shape
assert "offset" in parameter_names
assert "ignored_tensor" not in parameter_names
assert "offset" in state_keys
assert "ignored_tensor" not in state_keys

dict_keys(['offset'])


### Step 3: Register Persistent Non-Learnable State

In [ ]:
import torch
import torch.nn as nn

# Buffers store persistent non-learnable state (e.g., running stats, fixed masks).
# They are included in state_dict() and moved with the module, but not optimized.
# Use register_buffer() to declare them explicitly.
class FeatureCentering(nn.Module):
    def __init__(self, feature_size):
        super().__init__()
        self.register_buffer(
            "center", torch.zeros(feature_size)
        )

    def forward(self, x):
        return x - self.center

centering = FeatureCentering(feature_size=16)

assert dict(centering.named_parameters()) == {}
assert "center" in dict(centering.named_buffers())
assert "center" in centering.state_dict()

print("FeatureCentering named buffers:", dict(centering.named_buffers()).keys())

FeatureCentering named buffers: dict_keys(['center'])


### Step 4: Separate Module Mode from Gradient Tracking

In [ ]:
import torch
import torch.nn as nn

# Calling train() or eval() sets the module's mode for mode-dependent layers (Dropout, BatchNorm).
# This is independent of gradient recording: use both .eval() and no_grad()/inference_mode() for validation.
linear = nn.Linear(32, 32, bias=False)
features = torch.ones(32, requires_grad=True)

train_output = linear(features)
assert train_output.requires_grad

linear.eval()
with torch.inference_mode():
    eval_output = linear(features)
    assert not eval_output.requires_grad

assert linear.training is False
assert train_output.requires_grad
assert not eval_output.requires_grad

## 2. Creating Custom Layers & Activation Functions
### Step 1: Combine Stateful Modules with Stateless Functions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use nn modules for stateful operations (Linear, Conv2d, etc.).
# Use F.function for stateless computations (relu, gelu, dropout). F functions don't hold parameters.
class CustomMLPBlock(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)  # Common activation in modern Transformers
        x = self.fc2(x)
        return x

block = CustomMLPBlock(
    in_features=32,
    hidden_features=64,
    out_features=16,
)
sample = torch.randn(8, 32)
output = block(sample)

assert output.shape == (8, 16)
assert len(list(block.parameters())) == 4

## 3. Chaining Modules: `nn.Sequential` and `nn.ModuleList`
### Step 1: Build a Fixed Pipeline with `nn.Sequential`

In [6]:
import torch
import torch.nn as nn

deep_net = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 10),
)

dummy_input = torch.randn(32, 256)
predictions = deep_net(dummy_input)

assert predictions.shape == (32, 10)
assert len(list(deep_net.parameters())) == 4

### Step 2: Control Dynamic Execution with `nn.ModuleList`

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DynamicDepthNetwork(nn.Module):
    def __init__(self, num_layers, feature_dim):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(feature_dim, feature_dim)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            residual = x
            x = F.relu(layer(x))
            x = x + residual  # Skip connection
        return x

dynamic_net = DynamicDepthNetwork(
    num_layers=3,
    feature_dim=64,
)
sample = torch.randn(16, 64)
output = dynamic_net(sample)

assert output.shape == sample.shape
assert len(dynamic_net.layers) == 3
assert len(list(dynamic_net.parameters())) == 6

## 4. Gotchas & Reality Checks: Production Pitfalls
### Gotcha 1: The Standard Python List Trap

In [7]:
import torch.nn as nn

class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [
            nn.Linear(10, 10),
            nn.Linear(10, 10),
        ]

model = BrokenModel()
assert list(model.parameters()) == []

class RegisteredModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(10, 10),
            nn.Linear(10, 10),
        ])

registered_model = RegisteredModel()
assert len(list(registered_model.parameters())) == 4

### Gotcha 2: Weight Initialization, `.data`, and `no_grad()`

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomInitLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_dim, in_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim))

        # nn.init functions handle no_grad internally.
        nn.init.kaiming_normal_(
            self.weight,
            mode="fan_in",
            nonlinearity="relu",
        )

        # Raw in-place initialization requires an explicit context.
        with torch.no_grad():
            self.bias.fill_(0.01)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)

layer = CustomInitLayer(in_dim=16, out_dim=8)
sample = torch.randn(4, 16)
output = layer(sample)

assert output.shape == (4, 8)
assert torch.allclose(
    layer.bias,
    torch.full_like(layer.bias, 0.01),
)